# Phi 4 Model

# Imports

In [1]:
import gc

import LLMs, TP2
from LLMs import *
from TP2 import Dataset, RATIO_SPLIT_4C

C:\Users\Luco1421\Desktop\U\ia\TPs\TP2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Luco1421\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Charge model


In [2]:
LLMs.clean_gpu()

phi_4 = LLMs.charge_model("microsoft/Phi-4-mini-instruct")

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
Loading weights: 100%|██████████| 194/194 [00:00<00:00, 7834.01it/s]
C:\Users\Luco1421\Desktop\U\ia\TPs\TP2\.venv\Lib\site-packages\torch\nn\modules\module.py:1370: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return t.to(


## Function to request Phi 4's answer


In [3]:
def chat_phi_4(text: str, prompt: str):
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": text},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.1,
            top_p=0.1,
            do_sample=True
        )

    response_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    LLMs.clean_gpu()

    return extract_results(response_text)

### Example of use


In [10]:
tokenizer, model = phi_4
chat_phi_4(SAMPLE_1 + SAMPLE_2 + SAMPLE_3, CONTEXT + TEXT_BEGINNER)

NameError: name 'phi_4' is not defined

# Results

## Load dataset and test utils class

In [7]:
dataset = TP2.Dataset("FEINA_1.xlsx").read()
llm_utils = LLMs.TestLLMUtils()

# Test without Few Shots

In [7]:
llm_utils.test_without_shots("Phi 4", chat_phi_4, dataset)

KeyboardInterrupt: 

# Test with Few Shots

In [8]:
llm_utils.test_with_shots("Phi 4", chat_phi_4, dataset, [2, 4, 7])

Phi 4 with 2 shots - Accuracy: 0.49785539215686275
Phi 4 with 4 shots - Accuracy: 0.4958639705882353
Phi 4 with 7 shots - Accuracy: 0.508962770032174


# Test with all data

In [8]:
llm_utils.test_all_LLM("Phi 4", chat_phi_4, dataset, [2])

Phi 4 without shots: average = 0.4976, std = 0.0141
Phi 4 with 2 shots - average = 0.4899, std = 0.0240
--------------------------------------------------


{0: [0.4822485207100592,
  0.5075301204819277,
  0.4923780487804878,
  0.5060975609756098,
  0.5121951219512195,
  0.503725782414307,
  0.4915378955114054,
  0.5026335590669676,
  0.4868913857677903,
  0.526355421686747,
  0.5092165898617511,
  0.4860587792012057,
  0.5060240963855421,
  0.48618371919342795,
  0.516566265060241,
  0.5060975609756098,
  0.4873134328358209,
  0.4955357142857143,
  0.5201520912547528,
  0.4977409638554217,
  0.47023809523809523,
  0.46651785714285715,
  0.506859756097561,
  0.5063957863054929,
  0.49385560675883255,
  0.49097744360902257,
  0.4946646341463415,
  0.5072353389185073,
  0.47505584512285925,
  0.4950943396226415],
 2: [0.48298816568047337,
  0.509789156626506,
  0.48628048780487804,
  0.5053353658536586,
  0.4961890243902439,
  0.49254843517138597,
  0.49080206033848417,
  0.5052473763118441,
  0.4891385767790262,
  0.49623493975903615,
  0.5023041474654378,
  0.49736247174076864,
  0.5135542168674698,
  0.5041075429424944,
  0.48945783132530

In [9]:
LLMs.clean_gpu()

del phi_4, chat_phi_4, tokenizer, model
gc.collect()

1977